In [1]:
"""
This script is used to create the pycistopic object

authors: Roy Oelen, Martijn van der Werf

"""

'\nThis script is used to create the pycistopic object\n\nauthors: Roy Oelen, Martijn van der Werf\n\n'

In [2]:
# imports
from pycisTopic.cistopic_class import CistopicObject
from scipy.sparse import csr_matrix
from scipy.io import mmread
import gzip
from sklearn.preprocessing import binarize
import pandas as pd
import os
from pycisTopic.lda_models import run_cgs_models_mallet
from numpy.random import choice
from pycisTopic.lda_models import evaluate_models
import pickle
import numpy as np
import glob
import re


/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-01-23 17:22:15,141	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:
#######################################
# read the openness to nucleus matrix #
#######################################

# location of matrix
#fragment_matrix_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/matrix_merged.mtx.gz'
fragment_matrix_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/matrix_merged.mtx'
# open connection
#fragment_matrix_gzip = open(fragment_matrix_loc)
# read as coo matrix
#coo_fragment_matrix = mmread(fragment_matrix_gzip)
coo_fragment_matrix = mmread(fragment_matrix_loc)
# convert to csr format
fragment_matrix = csr_matrix(coo_fragment_matrix)
# close file handle
#fragment_matrix_gzip.close()

In [5]:
##########################################
# read nucleus barcodes and region names #
##########################################

# locations
barcodes_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/barcodes.tsv.gz'
features_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/features.tsv.gz'

# load region names
region_names_file = gzip.open(features_loc, 'rb')
region_names = pd.read_csv(region_names_file, sep='\t', header=None).iloc[:,0].to_list()
region_names = [i.replace('-', ':', 1) for i in region_names]

# Read cell names
cell_names_file = gzip.open(barcodes_loc, 'rb')
cell_names = pd.read_csv(cell_names_file, sep='\t', header=None).iloc[:,0].to_list()

In [6]:
#########################################
# set up path to all the fragment files #
#########################################

# fragment count cPeaks path
fragment_path = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/rounded_fragments/'
# List all files in the directory
fragment_files = os.listdir(fragment_path)
# compile a pattern
pattern = re.compile(r'.*tsv\.gz$')
# filter by pattern
fragment_files = [f for f in fragment_files if os.path.isfile(os.path.join(fragment_path, f)) and pattern.match(f)]
# order the files
fragment_files.sort()
# add the path
fragment_files = [os.path.join(fragment_path, f) for f in fragment_files]

In [7]:
#############################
# read the nucleus metadata #
#############################

# location of the metadata
cell_metadata_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/metadata.tsv.gz'
cell_metadata = pd.read_csv(cell_metadata_loc, sep = '\t', dtype = {'confined_condition': 'category', 'unconfined_condition': 'category','final_condition': 'category','LONG_COVID': 'category'})

# Subset cell metadata 
cell_metadata = cell_metadata[cell_metadata.bc.isin(cell_names)]
# Filter cell names
cell_names = cell_metadata.bc.to_list()
# Names as rownames 
cell_metadata.index = cell_metadata.bc

In [8]:
#####################################
# binarize the accessibility matrix #
#####################################

# binarization of accessibility matrix
fragment_matrix_binarized = binarize(fragment_matrix, threshold=0)

In [10]:
##########################################
# create metadata for all of the regions #
##########################################

# location of the cpeaks annotation
cpeaks_annotation_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/deconstruced_atac_objects/merged_major_celltypes/region_annotations.tsv.gz'
# read the annotations
cpeaks_annotation = pd.read_csv(cpeaks_annotation_loc, sep = '\t')
# add a name column
cpeaks_annotation['name'] = cpeaks_annotation['chr_hg38'].astype(str) + ':' + cpeaks_annotation['start_hg38'].astype(str) + '-' + cpeaks_annotation['end_hg38'].astype(str)

# Subset region metadata metadata 
cpeaks_annotation = cpeaks_annotation[cpeaks_annotation.name.isin(region_names)]
# Names as rownames 
cpeaks_annotation.index = cpeaks_annotation.name

In [11]:
#############################
# create pycistopic object #
#############################

# construct object
cistopic_obj = CistopicObject(
    fragment_matrix=fragment_matrix,
    binary_matrix=fragment_matrix_binarized,
    cell_names=cell_names,
    region_names=region_names,
    region_data=cpeaks_annotation,
    cell_data=cell_metadata,
    path_to_fragments=fragment_files,
    project='wijst_multiome'
)


In [12]:
##########################
# save pycistopic object #
##########################

# location to store the object
pycistopic_object_loc = '/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/objects/all_nuclei_and_regions.pkl'

# save the object
with open(pycistopic_object_loc, 'wb') as f:
   pickle.dump(cistopic_obj, f)
